# nb129 — Chemprop multi-task pretrain (pinned chemprop==2.0.4) + Tox21 fusion

Retry of nb127 with:
1. Pinned chemprop==2.0.4 (stable BatchMolGraph API; nb127 v4 hit 'TrainingBatch object has no attribute V' on latest version)
2. Add Tox21 binary NR labels (NR-AhR, NR-PPAR-gamma, NR-AR, NR-ER) as auxiliary multi-task heads — 6,000+ extra compounds with PXR-relevant labels
3. Force CPU (P100 GPU on Kaggle is sm_60, incompatible with bundled torch 2.5+)
4. Use Papyrus++ data (or full Papyrus if nb128 is done by the time this kernel runs)

Outputs: oof_nb129_chemprop.npy + te_nb129_chemprop.npy → ensemble candidates.

In [ ]:
import subprocess, sys, os, urllib.request
from pathlib import Path
os.environ['PYTHONUNBUFFERED'] = '1'

# Pin chemprop to a known-stable version
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'chemprop==2.0.4', 'rdkit', 'lightning'], check=False)
ACCEL = 'cpu'  # P100 incompatibility; force CPU

import torch
print(f'torch: {torch.__version__}')
import chemprop
print(f'chemprop: {chemprop.__version__}')

In [ ]:
# Find Papyrus parquet (prefer full from nb128, fall back to ++ from nb125)
import glob, pandas as pd, numpy as np

papy_path = None
for pat in [
    '/kaggle/input/**/papyrus_full_wide.parquet',
    '/kaggle/input/**/papyrus_wide_compound_x_target.parquet',
]:
    matches = glob.glob(pat, recursive=True)
    if matches:
        papy_path = matches[0]; print(f'Using: {papy_path}'); break

if not papy_path:
    print('No Papyrus parquet in dataset; downloading directly')
    subprocess.run([sys.executable,'-m','pip','install','-q','papyrus-scripts'],check=False)
    from papyrus_scripts import download_papyrus
    from papyrus_scripts.reader import read_papyrus
    from papyrus_scripts.preprocess import keep_accession
    download_papyrus(version='05.7', only_pp=True, structures=False, descriptors=None)
    TARGETS = ['O75469','Q14994','P11473','Q96RI1','Q13133','P55055','Q07869','P37231','Q03181',
               'P10276','P10826','P13631','P19793','P28702','P48443','P10275','P03372','Q92731',
               'P04150','P08235','P10827','P10828','P41235','P11474','O95718','P62508',
               'P08684','P11712','P33261','P05177','P10635','P05181','P20815','P20813',
               'P08183','Q9UNQ0','Q92887','Q9Y6L6','Q9NPD5','P35869','P02768']
    chunks = []
    for chunk in read_papyrus(version='05.7', plusplus=True, is3d=False, chunksize=200_000):
        sub = keep_accession(chunk, TARGETS)
        if len(sub)>0: chunks.append(sub)
    papy = pd.concat(chunks, ignore_index=True)
    smi_col = 'SMILES' if 'SMILES' in papy.columns else 'SMILES_Stripped'
    wide = papy.pivot_table(index=smi_col, columns='accession', values='pchembl_value_Mean', aggfunc='median').reset_index()
else:
    wide = pd.read_parquet(papy_path)

smi_col = 'SMILES' if 'SMILES' in wide.columns else ('SMILES_Stripped' if 'SMILES_Stripped' in wide.columns else 'connectivity')
target_cols = [c for c in wide.columns if c not in ('SMILES','SMILES_Stripped','connectivity','index')]
print(f'Wide papyrus: {wide.shape}  smi_col={smi_col}  n_targets={len(target_cols)}')
wide = wide.dropna(subset=[smi_col])

In [ ]:
# Load Tox21 from MoleculeNet — 6,000 compounds with NR + SR binary labels
import urllib.request
tox21_csv = '/kaggle/working/tox21_moleculenet.csv.gz'
if not Path(tox21_csv).exists():
    urllib.request.urlretrieve('https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz', tox21_csv)
tox21 = pd.read_csv(tox21_csv)
print(f'Tox21: {tox21.shape}  cols: {list(tox21.columns)[:10]}')
# Use NR columns (xenosensors PXR-related)
tox21_nr_cols = [c for c in tox21.columns if c.startswith('NR-')]
print(f'NR cols: {tox21_nr_cols}')

In [ ]:
# Combine: papyrus regression targets + tox21 binary NR labels
# Papyrus has continuous pchembl values; Tox21 has 0/1. Different heads.
# Merge on SMILES (after standardization).
from rdkit import Chem

def std_smi(s):
    try:
        m = Chem.MolFromSmiles(str(s))
        return Chem.MolToSmiles(m) if m is not None else None
    except Exception:
        return None

wide['std_smi'] = wide[smi_col].apply(std_smi)
wide = wide.dropna(subset=['std_smi'])
wide_p = wide.groupby('std_smi').first().reset_index()
tox21['std_smi'] = tox21['smiles'].apply(std_smi)
tox21 = tox21.dropna(subset=['std_smi'])

tox21_simple = tox21[['std_smi'] + tox21_nr_cols].groupby('std_smi').first().reset_index()
merged = pd.merge(wide_p, tox21_simple, on='std_smi', how='outer')
print(f'Merged: {merged.shape}')

y_pa_cols = target_cols
y_tx_cols = tox21_nr_cols
all_targets = y_pa_cols + y_tx_cols
n_pa = len(y_pa_cols); n_tx = len(y_tx_cols)
print(f'Pretrain heads: {n_pa} regression + {n_tx} binary = {n_pa+n_tx}')

y_combined = merged[all_targets].to_numpy(dtype=np.float32)  # NaN where missing
smiles_combined = merged['std_smi'].tolist()
print(f'Pretrain compounds: {len(smiles_combined)}  NaN frac: {np.isnan(y_combined).mean()*100:.1f}%')

In [ ]:
# Build Chemprop multi-target dataset
from chemprop import data, featurizers, models
from chemprop import nn as cnn
from lightning import pytorch as L

datapoints_pretrain = []
for s, y in zip(smiles_combined, y_combined):
    try:
        datapoints_pretrain.append(data.MoleculeDatapoint.from_smi(s, y))
    except Exception:
        pass
print(f'Built {len(datapoints_pretrain)} pretrain datapoints')

featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()
ds_pretrain = data.MoleculeDataset(datapoints_pretrain, featurizer)
loader_pretrain = data.build_dataloader(ds_pretrain, batch_size=128, num_workers=0, shuffle=True)
print('Dataset ready')

In [ ]:
# Pretrain — small/fast model for CPU
n_tasks = y_combined.shape[1]
mp = cnn.BondMessagePassing(depth=3, d_h=200, dropout=0.1)
agg = cnn.MeanAggregation()
ffn = cnn.RegressionFFN(n_tasks=n_tasks, input_dim=mp.output_dim, hidden_dim=200, n_layers=2, dropout=0.1)
model = models.MPNN(mp, agg, ffn, batch_norm=True, metrics=[cnn.metrics.MAE()])

trainer = L.Trainer(max_epochs=12, accelerator=ACCEL, devices=1, gradient_clip_val=1.0,
                    enable_progress_bar=False, logger=False, enable_checkpointing=False)
import time
t0 = time.time()
trainer.fit(model=model, train_dataloaders=loader_pretrain)
print(f'Pretrain done: {(time.time()-t0)/60:.1f} min')

In [ ]:
# Fine-tune on PXR with scaffold 5-fold CV
import urllib.request
HF = 'https://huggingface.co/datasets/openadmet/pxr-challenge-train-test/resolve/main'
TR_LOC = '/kaggle/working/train.csv'; TE_LOC = '/kaggle/working/test.csv'
if not Path(TR_LOC).exists():
    urllib.request.urlretrieve(f'{HF}/pxr-challenge_TRAIN.csv', TR_LOC)
    urllib.request.urlretrieve(f'{HF}/pxr-challenge_TEST_BLINDED.csv', TE_LOC)
tr = pd.read_csv(TR_LOC); te = pd.read_csv(TE_LOC)

pxr_dps_tr, valid_tr = [], []
for s, y in zip(tr['SMILES'], tr['pEC50']):
    if pd.isna(y) or pd.isna(s): valid_tr.append(False); continue
    try:
        pxr_dps_tr.append(data.MoleculeDatapoint.from_smi(s, np.array([y], dtype=np.float32)))
        valid_tr.append(True)
    except Exception: valid_tr.append(False)
pxr_dps_te = []
for s in te['SMILES']:
    if pd.isna(s): continue
    try:
        pxr_dps_te.append(data.MoleculeDatapoint.from_smi(s, np.array([0.0], dtype=np.float32)))
    except Exception: pass
print(f'PXR train: {len(pxr_dps_tr)}  test: {len(pxr_dps_te)}')

from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import defaultdict
tr_kept = tr[valid_tr].reset_index(drop=True)
scaffolds = []
for s in tr_kept['SMILES']:
    try:
        m = Chem.MolFromSmiles(s)
        scaffolds.append(MurckoScaffold.MurckoScaffoldSmiles(mol=m) if m else str(s))
    except Exception: scaffolds.append(str(s))
scaf2idx = defaultdict(list)
for i, s in enumerate(scaffolds): scaf2idx[s].append(i)
groups = sorted(scaf2idx.items(), key=lambda x: -len(x[1]))
fold_assign = np.zeros(len(scaffolds), dtype=int)
for fi, (scaf, idx) in enumerate(groups):
    for i in idx: fold_assign[i] = fi % 5
print('fold sizes:', [(fold_assign==i).sum() for i in range(5)])

oof = np.zeros(len(pxr_dps_tr), dtype=np.float32)
te_preds = []
for fold in range(5):
    print(f'\n--- Fold {fold+1}/5 ---')
    val_mask = (fold_assign == fold); tr_mask = ~val_mask
    dp_tr = [pxr_dps_tr[i] for i in range(len(pxr_dps_tr)) if tr_mask[i]]
    dp_va = [pxr_dps_tr[i] for i in range(len(pxr_dps_tr)) if val_mask[i]]
    ldr_tr = data.build_dataloader(data.MoleculeDataset(dp_tr, featurizer), batch_size=128, shuffle=True)
    ldr_va = data.build_dataloader(data.MoleculeDataset(dp_va, featurizer), batch_size=128, shuffle=False)
    ldr_te = data.build_dataloader(data.MoleculeDataset(pxr_dps_te, featurizer), batch_size=128, shuffle=False)
    ffn_pxr = cnn.RegressionFFN(n_tasks=1, input_dim=mp.output_dim, hidden_dim=200, n_layers=2, dropout=0.1)
    model_pxr = models.MPNN(mp, agg, ffn_pxr, batch_norm=True, metrics=[cnn.metrics.MAE()])
    trainer_ft = L.Trainer(max_epochs=8, accelerator=ACCEL, devices=1, gradient_clip_val=1.0,
                           enable_progress_bar=False, logger=False, enable_checkpointing=False)
    trainer_ft.fit(model=model_pxr, train_dataloaders=ldr_tr, val_dataloaders=ldr_va)
    pred_va = torch.cat([model_pxr(b).cpu() for b in ldr_va]).numpy().flatten()
    pred_te = torch.cat([model_pxr(b).cpu() for b in ldr_te]).numpy().flatten()
    oof[val_mask] = pred_va[:val_mask.sum()]
    te_preds.append(pred_te)
te_pred = np.mean(te_preds, axis=0)
y_arr = tr_kept['pEC50'].values
rae_v = np.abs(y_arr - oof).sum() / np.abs(y_arr - y_arr.mean()).sum()
ratio = te_pred.std() / oof.std()
print(f'\n=== OOF RAE={rae_v:.4f}  ratio={ratio:.3f}  te_std={te_pred.std():.3f}')
np.save('/kaggle/working/oof_nb129_chemprop.npy', oof)
np.save('/kaggle/working/te_nb129_chemprop.npy', te_pred)
print('saved')